# Week 02 - Image Representation and Colour Spaces

**MCTE 4323 / MCTA 4364 Machine Vision**

### Learning objectives
By the end of this lab you will be able to:
- Explain why colour is represented in different spaces (BGR, RGB, HSV, L\*a\*b\*).
- Convert between colour spaces and describe what each channel encodes.
- Segment an object by colour using `cv2.inRange` in HSV space.
- Interpret an image histogram and use it to choose thresholds.

### Key idea
A colour image is a 3D array `(H, W, 3)`. The *colour space* is a convention for organising those three numbers. BGR separates colour into primaries (bad for describing perception), while HSV separates **Hue** (which colour), **Saturation** (how pure) and **Value** (how bright), which makes many vision tasks easier.

## 1. Setup
Run once per Colab session (skip the clone if working locally).

In [ ]:
import os
if not os.path.isdir("MCTA-4364-Machine-Vision"):
    !git clone https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git
%cd MCTA-4364-Machine-Vision
!pip -q install opencv-python matplotlib numpy

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def show(*images, titles=None, cmap=None):
    titles = titles or [""] * len(images)
    plt.figure(figsize=(5 * len(images), 5))
    for i, img in enumerate(images):
        plt.subplot(1, len(images), i + 1)
        if img.ndim == 2:
            plt.imshow(img, cmap=cmap or "gray")
        elif img.shape[2] == 3:
            plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        else:
            plt.imshow(img)
        plt.title(titles[i]); plt.axis("off")
    plt.tight_layout(); plt.show()

img = cv2.imread("resources/images/messi.jpg")
print("Shape:", img.shape, "| dtype:", img.dtype)

## 2. Guided example - BGR vs RGB
OpenCV reads images as **BGR**. Matplotlib expects **RGB**. If you forget to convert, red and blue will be swapped.

In [ ]:
rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.imshow(img); plt.title("BGR shown as RGB (wrong)"); plt.axis("off")
plt.subplot(1, 2, 2); plt.imshow(rgb); plt.title("Correctly converted to RGB"); plt.axis("off")
plt.show()

## 3. Guided example - HSV colour space
HSV is powerful for colour segmentation. In OpenCV the ranges are:
- **H** (hue): 0-179
- **S** (saturation): 0-255
- **V** (value): 0-255

Hue is a *circular* quantity: red wraps around from 0 to 179.

In [ ]:
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
h, s, v = cv2.split(hsv)
show(h, s, v, titles=["Hue", "Saturation", "Value"], cmap="gray")

## 4. Guided example - colour-based segmentation with `inRange`

The football pitch is green. Green hue is roughly **35-85** in OpenCV. We build a binary mask with `cv2.inRange` and keep only those pixels.

In [ ]:
lower = np.array([35, 40, 40])
upper = np.array([85, 255, 255])

mask = cv2.inRange(hsv, lower, upper)
segmented = cv2.bitwise_and(img, img, mask=mask)

show(mask, segmented, titles=["Green mask", "Segmented result"], cmap="gray")

> **Observation:** the mask is not clean - there are holes and stray pixels. In Week 6 you will fix this with **morphology**, and in Week 7 with **contours**. Here we focus only on colour.

## 5. Guided example - histograms
A histogram counts how many pixels have each intensity. It tells you whether the image is dark, bright or low-contrast, which guides threshold choice.

In [ ]:
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1); plt.imshow(gray, cmap="gray"); plt.title("Grayscale"); plt.axis("off")
plt.subplot(1, 2, 2)
for i, colour in enumerate(["b", "g", "r"]):
    hist = cv2.calcHist([img], [i], None, [256], [0, 256])
    plt.plot(hist, color=colour)
plt.title("Per-channel histogram"); plt.xlabel("Intensity"); plt.xlim([0, 256]); plt.show()

## 6. Exercise (complete the code)

Build a **skin-colour detector**. Skin hue is roughly **0-20** (and wrap-around **160-179**) with moderate saturation.

1. Convert `resources/images/hasan.jpg` to HSV.
2. Build a mask for skin using two `inRange` calls (one for 0-20 and one for 160-179) and combine them with `cv2.bitwise_or`.
3. Display the mask and the segmented image.

*Hint:* `mask2 = cv2.inRange(hsv, np.array([160,40,40]), np.array([179,255,255]))`

In [ ]:
skin = cv2.imread("resources/images/hasan.jpg")
hsv_skin = cv2.cvtColor(skin, cv2.COLOR_BGR2HSV)

# TODO: mask_low = cv2.inRange(...)
# TODO: mask_high = cv2.inRange(...)
# TODO: skin_mask = cv2.bitwise_or(mask_low, mask_high)
# TODO: display with show(...)

## 7. Challenge (independent)

Implement a **green-screen effect**: replace all green pitch pixels in `messi.jpg` with a solid colour (e.g. white) to simulate background removal. 

- Use the mask from Section 4.
- Set `img[mask > 0] = [255, 255, 255]` (remember BGR order).
- Then invert the mask and try keeping only the background.

In [ ]:
# Your code here


## 8. Reflection
1. Why is colour segmentation usually easier in HSV than in BGR?
2. What does a single sharp spike in a histogram tell you about the image?
3. Give one engineering application where colour segmentation would fail.